In [ ]:
import sys, os
# notebook 位于 data/ 目录，xlsx_reader.py 在同目录，无需修改 sys.path
# 若从项目根目录运行则取消下行注释：
# sys.path.insert(0, os.path.abspath(".."))


## 1. 读取 xlsx 文件

In [9]:
from xlsx_reader import read_plc_project_xlsx, print_project_summary

XLSX_PATH = "PLC程序完整.xlsx"  # ← 如文件名不同请修改

spec = read_plc_project_xlsx(XLSX_PATH)
print_project_summary(spec)



设备清单（3 项）
  [cpu] SIMATIC s7-1200 1214C AC/DC/Rly 6ES7 214-1BG40-0XB0/V4.5  ×1  订货号:6ES7 214-1BG40-0XB0
  [通讯模块] CM 1241 (RS422/485) 6ES7 241-1CH32-0XB0  ×1  订货号:6ES7 241-1CH32-0XB0
  [数字输入量模块] DI 16x24VDC/DQ 16xRelay_1 6ES7 223-1PL30-0XB0  ×3  订货号:6ES7 223-1PL30-0XB0

I/O 变量表（98 个变量）
  [cpu点表] 25 个变量
  [数字输入量模块1] 32 个变量
  [数字输入量模块2] 31 个变量
  [数字输入量模块3] 10 个变量

DB 块（1 个）
  [风机输出功能] 25 个变量  描述长度:375 字符


## 2. 检验硬件设备清单

In [10]:
print(f"设备总数：{len(spec.hardware)}")
assert len(spec.hardware) > 0, "设备清单不能为空"

for hw in spec.hardware:
    assert hw.category, f"设备分类不能为空：{hw}"
    assert hw.model_full, f"型号不能为空：{hw}"
    assert hw.quantity >= 1, f"数量应 ≥ 1：{hw}"
    order = hw.order_number
    print(f"  [{hw.category}] 订货号={order}  ×{hw.quantity}")

# 确保至少有一个 CPU
cpu_list = [hw for hw in spec.hardware if "cpu" in hw.category.lower()]
assert len(cpu_list) >= 1, "至少应有一台 CPU"

print("✓ 硬件清单校验通过")


设备总数：3
  [cpu] 订货号=6ES7 214-1BG40-0XB0  ×1
  [通讯模块] 订货号=6ES7 241-1CH32-0XB0  ×1
  [数字输入量模块] 订货号=6ES7 223-1PL30-0XB0  ×3
✓ 硬件清单校验通过


## 3. 检验 I/O 点表

In [3]:
import re
from collections import Counter

print(f"I/O 变量总数：{len(spec.io_tags)}")
assert len(spec.io_tags) > 0, "I/O 点表不能为空"

# S7-1200 合法地址格式：%I0.0 / %Q0.0 / %M0.0 / %IB0 / %QW2 / %MD4 等
ADDR_RE = re.compile(r'^%[IQM]\d+\.\d+$|^%[IQM][BWDLX]\d+$', re.IGNORECASE)

addr_errors = []
for tag in spec.io_tags:
    assert tag.name, f"变量名不能为空：{tag}"
    assert tag.address, f"地址不能为空：{tag}"
    if not ADDR_RE.match(tag.address):
        addr_errors.append(f"  地址格式异常：{tag.address!r}  ({tag.name})")

if addr_errors:
    print(f"⚠ 地址格式问题（{len(addr_errors)} 处）：")
    for e in addr_errors:
        print(e)
else:
    print("✓ 所有地址格式合法")

# 按模块统计变量数量
module_counts = Counter(t.module_group for t in spec.io_tags)
for mod, cnt in sorted(module_counts.items()):
    print(f"  [{mod}] {cnt} 个变量")

# 确保输入输出点均存在
input_cnt = sum(1 for t in spec.io_tags if t.address.upper().startswith("%I"))
output_cnt = sum(1 for t in spec.io_tags if t.address.upper().startswith("%Q"))
assert input_cnt > 0, "输入点（%I）为空"
assert output_cnt > 0, "输出点（%Q）为空"
print(f"  输入点 %I：{input_cnt} 个，输出点 %Q：{output_cnt} 个")

print("✓ I/O 点表校验通过")


I/O 变量总数：98
✓ 所有地址格式合法
  [cpu点表] 25 个变量
  [数字输入量模块1] 32 个变量
  [数字输入量模块2] 31 个变量
  [数字输入量模块3] 10 个变量
  输入点 %I：51 个，输出点 %Q：47 个
✓ I/O 点表校验通过


## 4. 检验 DB 块

In [4]:
from xlsx_reader import DBBlockSpec, DBVariableEntry

# 已知合法的变量组名（与 xlsx 列组标题一致）
VALID_GROUPS = {"信号数据", "数字量信号", "内部数据", "通讯数据"}

print(f"DB 块总数：{len(spec.db_blocks)}")
assert len(spec.db_blocks) > 0, "DB 块不能为空"

for db in spec.db_blocks:
    assert db.function_name, f"DB 功能名不能为空：{db}"
    assert len(db.variables) > 0, f"DB 块 '{db.function_name}' 变量列表为空"
    assert db.description, f"DB 块 '{db.function_name}' 功能描述为空"

    group_error = []
    for var in db.variables:
        assert var.name, f"变量名不能为空（块: {db.function_name}）"
        assert var.data_type, f"变量类型不能为空：{var.name}（块: {db.function_name}）"
        if var.group not in VALID_GROUPS:
            group_error.append(f"    未知组名 {var.group!r}：{var.name}")

    status = "⚠" if group_error else "✓"
    print(f"\n{status} [{db.function_name}]  {len(db.variables)} 个变量")
    print(f"  描述（前80字）：{db.description[:80]}")

    # 按组统计
    from collections import Counter
    gc = Counter(v.group for v in db.variables)
    for grp, cnt in sorted(gc.items()):
        print(f"    [{grp}] {cnt} 个变量")

    if group_error:
        print("  组名异常：")
        for e in group_error:
            print(e)

print("\n✓ DB 块校验通过")


DB 块总数：1

✓ [风机输出功能]  25 个变量
  描述（前80字）：当按下风机运行按钮，且风机无故障反馈时，风机运行标志置 ON，同时系统会先判断控制模式：如果是自动模式，就直接将预设的风机自动输出作为风机控制输出；如果是手动模
    [信号数据] 8 个变量
    [内部数据] 9 个变量
    [数字量信号] 4 个变量
    [通讯数据] 4 个变量

✓ DB 块校验通过


## 5. 快速查询示例

In [7]:
# ── 5-1 按地址前缀筛选输入/输出点 ────────────────────────────────────────────
input_tags  = [t for t in spec.io_tags if t.address.upper().startswith("%I")]
output_tags = [t for t in spec.io_tags if t.address.upper().startswith("%Q")]
print(f"输入点（%I）：{len(input_tags)} 个")
for t in input_tags[:5]:
    print(f"  {t.address:10s}  {t.name}  [{t.module_group}]")
print(f"  ...")
print(f"输出点（%Q）：{len(output_tags)} 个")
for t in output_tags[:5]:
    print(f"  {t.address:10s}  {t.name}  [{t.module_group}]")
print()

# ── 5-2 按关键字搜索 DB 变量 ─────────────────────────────────────────────────
KEYWORD = "反馈"  # ← 可自定义关键字
matched = [
    (db.function_name, var)
    for db in spec.db_blocks
    for var in db.variables
    if KEYWORD in var.name
]
print(f"含 '{KEYWORD}' 的 DB 变量（共 {len(matched)} 个）：")
for fn, var in matched:
    print(f"  [{fn}][{var.group}] {var.name}  {var.data_type}  offset={var.offset}")
print()

# ── 5-3 打印完整 DB 变量表 ────────────────────────────────────────────────────
for db in spec.db_blocks:
    print(f"\n{'─'*60}")
    print(f"DB 块：{db.function_name}")
    print(f"{'─'*60}")
    print(f"  {'名称':<20} {'类型':<25} {'偏移量':<10} 组")
    print(f"  {'─'*20} {'─'*25} {'─'*10} {'─'*12}")
    for var in db.variables:
        print(f"  {var.name:<20} {var.data_type:<25} {str(var.offset):<10} {var.group}")
print()

# ── 5-4 检验并打印功能描述 ───────────────────────────────────────────────────
print(f"{'='*60}")
print("功能描述校验")
print(f"{'='*60}")
for db in spec.db_blocks:
    desc = db.description
    assert desc, f"DB 块 '{db.function_name}' 功能描述为空"
    # 描述应包含实质内容（非全空白、长度合理）
    assert len(desc.strip()) >= 10, \
        f"DB 块 '{db.function_name}' 功能描述过短（{len(desc.strip())} 字符），疑似解析失败"
    print(f"\n【{db.function_name}】  共 {len(desc)} 字符")
    # 按换行符分段打印，保留原始段落结构
    for para in desc.split("\n"):
        para = para.strip()
        if para:
            # 每段最多显示 80 字符，超出则截断并标注
            if len(para) > 80:
                print(f"  {para[:80]}…（共{len(para)}字）")
            else:
                print(f"  {para}")
print(f"\n✓ 功能描述校验通过（共 {len(spec.db_blocks)} 个 DB 块）")


print(db.description)

输入点（%I）：51 个
  %I0.0       1#小火反馈  [cpu点表]
  %I2.0       5#大火反馈  [数字输入量模块1]
  %I4.0       11#小火反馈  [数字输入量模块2]
  %I6.2       消音复位  [数字输入量模块3]
  %I0.1       1#故障反馈  [cpu点表]
  ...
输出点（%Q）：47 个
  %Q6.0       15#小火  [数字输入量模块3]
  %Q6.1       15#复位  [数字输入量模块3]
  %Q6.2       15#大火  [数字输入量模块3]
  %Q6.5       烧嘴电源  [数字输入量模块3]
  %Q6.6       蜂鸣器  [数字输入量模块3]

含 '反馈' 的 DB 变量（共 5 个）：
  [风机输出功能][通讯数据] 变频器通讯反馈频率  Int  offset=34
  [风机输出功能][数字量信号] 风机运行反馈  Bool  offset=10.2
  [风机输出功能][数字量信号] 风机故障反馈  Bool  offset=10.3
  [风机输出功能][信号数据] 风机反馈频率  Real  offset=142
  [风机输出功能][内部数据] 风机反馈数据组  Array[0..15] of Bool  offset=138


────────────────────────────────────────────────────────────
DB 块：风机输出功能
────────────────────────────────────────────────────────────
  名称                   类型                        偏移量        组
  ──────────────────── ───────────────────────── ────────── ────────────
  风压故障                 Bool                      129.2      信号数据
  风机运行标志               Bool                      10         数

In [8]:
# ── 6. 用模板文件验证解析器 ───────────────────────────────────────────────────
# 运行 create_template.py 生成 PLC程序模板.xlsx（已预先生成，可直接跳过此行）
# import subprocess; subprocess.run(["python", "create_template.py"], check=True)

spec_tpl = read_plc_project_xlsx("PLC程序模板.xlsx")
print_project_summary(spec_tpl)

# 对模板做全量校验
from xlsx_reader import DBBlockSpec, DBVariableEntry
VALID_GROUPS = {"信号数据", "数字量信号", "内部数据", "通讯数据"}

assert len(spec_tpl.hardware) == 3
assert len(spec_tpl.io_tags)  == 35
assert len(spec_tpl.db_blocks) == 2

for db in spec_tpl.db_blocks:
    assert db.function_name
    assert len(db.variables) > 0
    assert len(db.description.strip()) >= 10, f"描述过短：{db.function_name}"
    for var in db.variables:
        assert var.name
        assert var.data_type
        assert var.group in VALID_GROUPS, f"组名异常 {var.group!r}：{var.name}"

print("\n✓ 模板文件解析校验全部通过")



设备清单（3 项）
  [cpu] SIMATIC S7-1200 1214C AC/DC/Rly 6ES7 214-1BG40-0XB0  ×1  订货号:6ES7 214-1BG40-0XB0
  [通讯模块] CM 1241 (RS422/485) 6ES7 241-1CH32-0XB0  ×1  订货号:6ES7 241-1CH32-0XB0
  [数字输入量模块] DI 16x24VDC/DQ 16xRelay 6ES7 223-1PL30-0XB0  ×2  订货号:6ES7 223-1PL30-0XB0

I/O 变量表（35 个变量）
  [cpu点表] 11 个变量
  [数字输入量模块1] 12 个变量
  [数字输入量模块2] 12 个变量

DB 块（2 个）
  [风机输出功能] 25 个变量  描述长度:228 字符
  [燃烧器控制功能] 12 个变量  描述长度:98 字符

✓ 模板文件解析校验全部通过
